# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded, get_data_compacted

In [2]:
nome_do_arquivo = 'kmodels.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    models = exp['model']

del exp
del arquivo

In [3]:
data = Experiment_Data()

data.load(path='../testing_data.csv')

expansions = {
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    'p': ['p0', 'p1'],
}

df = get_data_expanded(data.build_training_dataset(), expansions)
# df = df.loc[df['episode']<15].copy()
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,s0,s1,s2,s3,s_0,s_1,s_2,s_3,p0,p1
1866,18,92,"(0.2225177737903001, 0.4165106988169785)","(0.034, -0.408, -0.163, 0.032)",0,1.0,"(0.026, -0.604, -0.162, 0.343)",1.0,1.0,"(0.014, -0.402, -0.156, -0.103)",0.034,-0.408,-0.163,0.032,0.026,-0.604,-0.162,0.343,0.222518,0.416511
1652,43,82,"(0.2399377515223238, 0.3748028622994311)","(-0.286, -1.311, 0.152, 1.688)",1,1.0,"(-0.312, -1.113, 0.186, 1.33)",0.0,1.0,"(-0.335, -1.317, 0.212, 1.84)",-0.286,-1.311,0.152,1.688,-0.312,-1.113,0.186,1.330,0.239938,0.374803
612,27,31,"(0.4456006051678612, 0.2809654390334451)","(0.053, -0.245, -0.048, 0.435)",1,1.0,"(0.048, -0.03, -0.039, -0.296)",0.0,1.0,"(0.047, -0.242, -0.045, 0.37)",0.053,-0.245,-0.048,0.435,0.048,-0.030,-0.039,-0.296,0.445601,0.280965
2063,34,98,"(0.1955406148449595, 0.6245501454716056)","(0.016, -0.326, -0.109, -0.073)",0,1.0,"(0.01, -0.518, -0.111, 0.147)",0.0,1.0,"(-0.001, -0.71, -0.108, 0.367)",0.016,-0.326,-0.109,-0.073,0.010,-0.518,-0.111,0.147,0.195541,0.624550
1907,24,93,"(0.1059562422360733, 0.325731494914702)","(0.009, 0.388, 0.1, -0.45)",0,1.0,"(0.017, 0.184, 0.091, 0.047)",0.0,1.0,"(0.021, -0.02, 0.092, 0.54)",0.009,0.388,0.100,-0.450,0.017,0.184,0.091,0.047,0.105956,0.325731


# Predict 

In [4]:


def optim_params(prediction_dataset):
    weights = prediction_dataset.copy()
    for m, _ in enumerate(models):
        weights['estimated_s'] = prediction_dataset[f'estimated_s_model_{m}']
        expansions = {f'estimated_s': [f'estimated_s0', f'estimated_s1', f'estimated_s2', f'estimated_s3']}
        data = get_data_expanded(weights, expansions)
        weights[f'estimated_s_model_{m}'] = data['estimated_s']
        for d in range(4):
            weights[f'estimated_s{d}_model{m}'] = data[f'estimated_s{d}']
    weights['A'] = weights.apply(lambda row: np.array([[row[f's_{d}'] for _,_ in enumerate(models)] for d in range(4)]),axis=1)
    weights['b'] = weights.apply(lambda row: np.array([row[f's__{d}'] for d in range(4)]),axis=1)
    weights['w'] = weights.apply(lambda row: np.linalg.lstsq(row.A, row.b)[0],axis=1)

    weights[[f'estimated_weight_{m}' for m, _ in enumerate(models)]] = weights.apply(lambda row: pd.Series(row['w']),axis=1)
    return weights

def predict_with_params(prediction_dataset, weights):
    expansions = {
        f'estimated_s_model_{m}': [f'estimated_s{d}_model{m}' for d in range(4)] for m,_ in enumerate(models)}
    df = get_data_expanded(prediction_dataset, expansions)
    for d in range(4):
        weights[f'estimated_weighted_s{d}'] = np.sum([df[f'estimated_s{d}_model{m}']* weights[f'estimated_weight_{m}'] for m, _ in enumerate(models)])
    weights = get_data_compacted(weights, {'estimated_weighted_s': [f'estimated_weighted_s{d}' for d in range(4)]})
    
    for m, _ in enumerate(models):
        prediction_dataset[f'weight_{m}'] = weights[f'estimated_weight_{m}']
    prediction_dataset['estimated_r'] = weights['estimated_r_model_0']
    prediction_dataset['estimated_s'] = weights['estimated_weighted_s']
    
    results = data.get_evaluation_metrics(prediction_dataset, p=False)
    prediction_dataset[f'rse'] = results['rse']
    prediction_dataset[f'rse_normalized'] = results['rse_normalized']

    prediction_dataset[f'rse_s0'] = results['rse_s0']
    prediction_dataset[f'rse_s1'] = results['rse_s1']
    prediction_dataset[f'rse_s2'] = results['rse_s2']
    prediction_dataset[f'rse_s3'] = results['rse_s3']

    prediction_dataset[f'rse_s0_normalized'] = results['rse_s0_normalized']
    prediction_dataset[f'rse_s1_normalized'] = results['rse_s1_normalized']
    prediction_dataset[f'rse_s2_normalized'] = results['rse_s2_normalized']
    prediction_dataset[f'rse_s3_normalized'] = results['rse_s3_normalized']

    return prediction_dataset

In [5]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_r_model_{i}'] = pred['estimated_r']
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [6]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_', 's_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3', 's__']] = pre_df[['s', 's0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3', 's_']]

    prediction_dataset = evaluate(models, pre_df)
    params = optim_params(prediction_dataset)
    prediction_dataset = evaluate(models, df)
    final_predictions = predict_with_params(prediction_dataset, params)

    cols = [
        'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ]

    return final_predictions[cols]

In [7]:
prediction_dataset = predicts(models, df)
prediction_dataset.head()

,estimated_s,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
1866,"(-12.388858153567678, -26.557514488546786, -0....",136.656465,7.559751,12.402858,26.155514,0.134393,97.963699,13.642934,6.955633,0.823486,8.816950
1652,"(-12.388858153567678, -26.557514488546786, -0....",137.434678,7.512430,12.053858,25.240514,0.233607,99.906699,13.274401,6.730707,1.061407,8.983203
612,"(-12.388858153567678, -26.557514488546786, -0....",137.211465,7.521866,12.435858,26.315514,0.023393,98.436699,13.677781,6.994964,0.557299,8.857423
2063,"(-12.388858153567678, -26.557514488546786, -0....",136.755465,7.518139,12.387858,25.847514,0.086393,98.433699,13.627094,6.879920,0.708378,8.857166
1907,"(-12.388858153567678, -26.557514488546786, -0....",137.667678,7.586367,12.409858,26.537514,0.113607,98.606699,13.650325,7.049537,0.773637,8.871969
